# Gaussian Splatting Colab

Run this notebook with a GPU runtime. It installs the repository dependencies, prepares images from Drive or uploads, runs COLMAP, trains, renders, and optionally copies outputs back to Drive.


In [1]:
# Runtime and project configuration
REPO_URL = "https://github.com/IlhyeonChoo/gaussian-splatting.git"
BRANCH = "main"
WORKDIR = "/content/gaussian-splatting"

SCENE_NAME = "room_410"
MODEL_PATH = f"output/{SCENE_NAME}_colab"

# Training defaults for Colab. Increase ITERATIONS for final runs.
ITERATIONS = 30000
MAX_TRAIN_CAMERAS = 0
CAMERA_QUALITY_RATIO = 0.7
CAMERA_SELECTION_SEED = 42
RESOLUTION = 1
DATA_DEVICE = "cuda"
EVAL_SPLIT = False
USE_SPARSE_ADAM = False

# Persistent storage.
MOUNT_DRIVE = True
DRIVE_PROJECT_DIR = f"/content/drive/MyDrive/3DGS/{SCENE_NAME}"
COPY_OUTPUT_TO_DRIVE = True


In [2]:
# Check GPU runtime.
import subprocess

subprocess.run(["nvidia-smi"], check=False)


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [3]:
import os
import subprocess
from pathlib import Path

repo_path = Path(WORKDIR)
if repo_path.exists() and (repo_path / ".git").exists():
    subprocess.run(["git", "-C", str(repo_path), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_path), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_path), "pull", "--ff-only", "origin", BRANCH], check=True)
elif repo_path.exists():
    raise RuntimeError(f"WORKDIR exists but is not a git repository: {repo_path}")
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo_path)], check=True)

os.chdir(repo_path)
try:
    subprocess.run(["bash", "colab/setup_colab.sh"], check=True, capture_output=True, text=True)
except subprocess.CalledProcessError as e:
    print(f"Error running setup script: {e}")
    print(f"Stdout:\n{e.stdout}")
    print(f"Stderr:\n{e.stderr}")
    raise # Re-raise the exception after printing details

In [4]:
# Mount Google Drive for input/output persistence.
import os
from pathlib import Path

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_PROJECT_DIR).mkdir(parents=True, exist_ok=True)
    print(f"Drive project directory: {DRIVE_PROJECT_DIR}")


Mounted at /content/drive
Drive project directory: /content/drive/MyDrive/3DGS/room_410


## Prepare Data

Set `INPUT_PATH` to a Drive folder, zip, video, or single image. For multiple uploaded images, use the upload cell below instead.


In [5]:
# Prepare a scene from a Drive/local path.
import os
import shutil
import sys
from pathlib import Path

os.chdir(WORKDIR)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

from colab.colab_utils import prepare_uploaded_path

# Examples:
# INPUT_PATH = "/content/drive/MyDrive/my_scene_images"
# INPUT_PATH = "/content/drive/MyDrive/my_scene_images.zip"
# INPUT_PATH = "/content/drive/MyDrive/my_scene_video.mp4"
INPUT_PATH = f"/content/drive/MyDrive/3DGS/data/{SCENE_NAME}.mp4"

MAX_INPUT_IMAGES = 10000
VIDEO_TARGET_FPS = 2.0
VIDEO_SCALE = 1

if INPUT_PATH:
    scene_path = prepare_uploaded_path(
        INPUT_PATH,
        SCENE_NAME,
        max_images=MAX_INPUT_IMAGES,
        root=WORKDIR,
        target_fps=VIDEO_TARGET_FPS,
        scale=VIDEO_SCALE,
    )
    print(f"Prepared scene: {scene_path}")
else:
    print("Set INPUT_PATH above, or run the upload cell below.")


[DONE] Prepared 464 image(s): /content/gaussian-splatting/data/room_410/input
Prepared scene: /content/gaussian-splatting/data/room_410


In [6]:
# Optional: upload images, a zip, or a video from your computer.
import os
import sys
from pathlib import Path

from google.colab import files

os.chdir(WORKDIR)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

from colab.colab_utils import IMAGE_EXTS, prepare_uploaded_path

uploaded = files.upload()
upload_dir = Path("/content/gaussian_splatting_uploads")
if upload_dir.exists():
    shutil.rmtree(upload_dir)
upload_dir.mkdir(parents=True, exist_ok=True)

saved_paths = []
for filename, content in uploaded.items():
    destination = upload_dir / filename
    destination.write_bytes(content)
    saved_paths.append(destination)

if saved_paths:
    all_images = all(path.suffix.lower() in IMAGE_EXTS for path in saved_paths)
    source = upload_dir if all_images else saved_paths[0]
    scene_path = prepare_uploaded_path(
        source,
        SCENE_NAME,
        max_images=MAX_INPUT_IMAGES,
        root=WORKDIR,
        target_fps=VIDEO_TARGET_FPS,
        scale=VIDEO_SCALE,
    )
    print(f"Prepared scene: {scene_path}")


In [7]:
# Verify prepared input images.
from pathlib import Path

scene_path = Path(WORKDIR) / "data" / SCENE_NAME
input_dir = scene_path / "input"
input_images = sorted(path for path in input_dir.glob("*") if path.is_file())
print(f"Scene path: {scene_path}")
print(f"Input images: {len(input_images)}")
print("First files:", [path.name for path in input_images[:5]])
if not input_images:
    raise RuntimeError("No input images found. Prepare data before continuing.")


Scene path: /content/gaussian-splatting/data/room_410
Input images: 464
First files: ['image_000000.jpg', 'image_000001.jpg', 'image_000002.jpg', 'image_000003.jpg', 'image_000004.jpg']


## COLMAP Conversion

This creates the `images/` and `sparse/0/` structure required by training.


In [8]:
import os
import subprocess

os.chdir(WORKDIR)
RUN_COLMAP = True
COLMAP_DEVICE = "auto"

if RUN_COLMAP:
    subprocess.run([
        "python", "convert.py",
        "-s", f"data/{SCENE_NAME}",
        "--colmap_device", COLMAP_DEVICE,
    ], check=True)


## Train

For smoke tests, keep `ITERATIONS` low. For real runs, increase it after confirming setup and COLMAP conversion work.


In [9]:
import os
import subprocess

os.chdir(WORKDIR)
RUN_TRAINING = True

if RUN_TRAINING:
    command = [
        "python", "train.py",
        "-s", f"data/{SCENE_NAME}",
        "-m", MODEL_PATH,
        "--iterations", str(ITERATIONS),
        "--max_train_cameras", str(MAX_TRAIN_CAMERAS),
        "--camera_quality_ratio", str(CAMERA_QUALITY_RATIO),
        "--camera_selection_seed", str(CAMERA_SELECTION_SEED),
        "--resolution", str(RESOLUTION),
        "--data_device", DATA_DEVICE,
        "--disable_viewer",
    ]
    if EVAL_SPLIT:
        command.append("--eval")
    if USE_SPARSE_ADAM:
        command.extend(["--optimizer_type", "sparse_adam"])

    print(" ".join(command))
    subprocess.run(command, check=True)


python train.py -s data/room_410 -m output/room_410_colab --iterations 30000 --max_train_cameras 0 --camera_quality_ratio 0.7 --camera_selection_seed 42 --resolution 1 --data_device cuda --disable_viewer


## Render and Save Outputs


In [ ]:
import os
import subprocess

os.chdir(WORKDIR)
RUN_RENDER = True
RUN_METRICS = False

if RUN_RENDER:
    subprocess.run(["python", "render.py", "-m", MODEL_PATH], check=True)

if RUN_METRICS:
    subprocess.run(["python", "metrics.py", "-m", MODEL_PATH], check=True)


In [11]:
# Copy the model directory to Drive for persistence.
import os
import sys

os.chdir(WORKDIR)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

from colab.colab_utils import copy_model_to_drive

if MOUNT_DRIVE and COPY_OUTPUT_TO_DRIVE:
    copy_model_to_drive(MODEL_PATH, DRIVE_PROJECT_DIR, overwrite=True)


[DONE] Copied model to: /content/drive/MyDrive/3DGS/room_410/room_410_colab
